# Sentiment Analysis Pipeline — Aspect Extraction: Keyword Matching vs spaCy

This notebook compares two approaches to aspect detection  
and introduces opinion extraction using spaCy's dependency parser.

The previous notebook established a fast, batch-optimized pipeline.  
The remaining limitation was the aspect detection step,  
which relied on simple keyword matching against a fixed list.

This notebook addresses that by introducing spaCy as an NLP layer,  
evaluating whether it improves aspect detection,  
and exploring what additional information the dependency tree can provide.

In [50]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from scipy.special import softmax
import urllib.request
import numpy as np
import pandas as pd
import torch
import time
import csv
import re
import spacy
nlp = spacy.load("en_core_web_sm")

/home/nhx/miniconda/envs/ai_app/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [51]:
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = "@user" if t.startswith("@") and len(t) > 1 else t
        t = "http" if t.startswith("http") else t
        new_text.append(t)
    return " ".join(new_text)

In [52]:
task = "sentiment"
MODEL = f"cardiffnlp/twitter-roberta-base-{task}"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

labels = []
mapping_link = f"https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets/{task}/mapping.txt"

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3580.67it/s]


In [53]:
with urllib.request.urlopen(mapping_link) as f:
    html = f.read().decode("utf-8").split("\n")
    csvreader = csv.reader(html, delimiter="\t")
    labels = [row[1] for row in csvreader if len(row) > 1]

In [54]:
def split_into_phrases(text):
    parts = re.split(r"\s*(?:\.|\bbut\b|\bhowever\b)\s*",text,flags=re.IGNORECASE)
    return [p.strip() for p in parts if p.strip()]

In [60]:
ASPECTS = ["design", "performance", "price", "service", "quality", "delivery", "staff"]

## 2. Aspect Detection — Two Approaches

We define two separate functions for aspect detection  
to allow a direct comparison on the same inputs.

**Approach 1 — spaCy lemma matching**  
Uses spaCy to tokenize the phrase and matches each token's lemma  
against the aspect list.  
The lemma is the base form of a word, so inflected forms such as  
"prices", "deliveries", or "designed" are reduced before matching.

**Approach 2 — Keyword matching**  
The approach used in the previous notebooks.  
Checks whether the aspect string appears anywhere inside the phrase string.  
No linguistic processing — purely substring-based.

In [73]:
def get_aspects_lemma(phrases : str):
    doc = nlp(phrases)
    found_aspects = []
    for token in doc:
        if token.lemma_ in ASPECTS:
            found_aspects.append(token.lemma_.lower())
    return found_aspects
    

In [74]:
def get_aspects(phrase):
    found_aspects = []
    for aspect in ASPECTS:
        if aspect.lower() in phrase.lower():
            found_aspects.append(aspect.lower())
    return found_aspects

In [55]:
model.eval()
def predict_batch(texts):
    texts = [preprocess(t) for t in texts]
    encoded_input = tokenizer(texts,return_tensors="pt",padding=True,truncation=True,max_length=128)

    with torch.no_grad():
        output = model(**encoded_input)

    scores = output.logits.detach().numpy()
    scores = softmax(scores, axis=1)
    results = []

    for score in scores:
        ranking = np.argsort(score)[::-1]
        label = labels[ranking[0]]
        confidence = round(float(score[ranking[0]]), 2)

        results.append({"Label": label,"Score": confidence})

    return results

In [76]:
def analyze_comment_batch(text):
    results_with_aspects = []
    results_without_aspects = []

    all_phrases = []

    for t in text:
        phrase = split_into_phrases(t)
        all_phrases.extend(phrase)

    predictions = predict_batch(all_phrases)

    for phrase, pred in zip(all_phrases, predictions):
        aspects = get_aspects(phrase)

        if aspects:
            results_with_aspects.append({"Phrase": phrase,"Aspects": aspects,"Label": pred["Label"],"Score": pred["Score"]})
        else:
            results_without_aspects.append({"Phrase": phrase,"Label": pred["Label"],"Score": pred["Score"]})

    return {
        "with_aspects": results_with_aspects,
        "without_aspects": results_without_aspects
    }

In [75]:
def analyze_comment_batch_lemma(text):
    results_with_aspects = []
    results_without_aspects = []

    all_phrases = []

    for t in text:
        phrase = split_into_phrases(t)
        all_phrases.extend(phrase)

    predictions = predict_batch(all_phrases)

    for phrase, pred in zip(all_phrases, predictions):
        aspects = get_aspects_lemma(phrase)

        if aspects:
            results_with_aspects.append({"Phrase": phrase,"Aspects": aspects,"Label": pred["Label"],"Score": pred["Score"]})
        else:
            results_without_aspects.append({"Phrase": phrase,"Label": pred["Label"],"Score": pred["Score"]})

    return {
        "with_aspects": results_with_aspects,
        "without_aspects": results_without_aspects
    }

## 3. Designing the Test Set

The test set is specifically constructed to expose the differences  
between the two methods rather than simply testing general performance.

It covers six categories:

- **Plurals and inflections** — e.g. "prices", "deliveries", "designs" — keyword matching misses these, lemma matching should not
- **Verb forms** — e.g. "delivered", "staffed" — tests whether the lemmatizer reduces them correctly
- **Substring false positives** — e.g. "priceless", "self-service", "staffroom" — the aspect keyword appears inside a longer word with a different meaning
- **Opinion-linked inputs** — phrases where a descriptive adjective is clearly tied to an aspect, used later for opinion extraction
- **Negation** — phrases where the opinion is negated, e.g. "not good", "never bad"
- **No-aspect inputs** — to confirm that neither method fires on aspect-free phrases

In [68]:
test_comments = [
    # --- PLURALS: old method misses, spaCy catches via lemma ---
    # "deliveries" → lemma "delivery" ✓
    "The deliveries were always on time.",
    # "prices" → lemma "price" ✓
    "The prices are way too high.",
    # "designs" → lemma "design" ✓
    "Their designs have really improved lately.",

    # --- VERB FORMS: old method misses, spaCy catches via lemma ---
    # "performing" → lemma "performance" ✗ (old misses), spaCy catches via POS+lemma
    "The device started performing much better after the update.",
    # "delivered" → lemma "delivery" ✓
    "The package was delivered in perfect condition.",
    # "staffed" → lemma "staff" ✓
    "The store was well staffed during peak hours.",

    # --- SUBSTRING FALSE POSITIVES: old method wrongly matches, spaCy does not ---
    # "price" inside "priceless" — old method hits, spaCy checks token boundary
    "The experience was absolutely priceless.",
    # "service" inside "self-service" — old method hits
    "They offer a self-service checkout option.",
    # "performance" inside "performances" already handled but also:
    # "staff" inside "staffroom" — old method hits
    "There is a staffroom on the second floor.",

    # --- OPINION EXTRACTION: old returns aspect only, spaCy returns aspect + opinion ---
    # Both catch "service", but spaCy tells you it was "slow"
    "The service was painfully slow.",
    # Both catch "design", but only spaCy tells you it was "beautiful"
    "The design is absolutely beautiful.",
    # Both catch "quality", but only spaCy tells you it was "poor"
    "The quality was surprisingly poor.",

    # --- NEGATION: old is blind to it, spaCy can detect the negator ---
    # "service" found by both, but spaCy sees "not" + "good" = negated positive
    "The service was not good at all.",
    # "performance" found by both, but spaCy sees "never" + "bad" = negated negative
    "The performance was never bad in my experience.",

    # --- NO ASPECT: both should return empty, proving neither over-fires ---
    "I really enjoyed the whole experience.",
    "Absolutely worth every penny.",
    "Would not recommend this to anyone.",
]

In [77]:
def get_results(func , comments):
    start_time = time.perf_counter()
    result = func(comments)
    end_time = time.perf_counter()
    print(f"This batch function took {end_time - start_time} s to be executed")
    df_aspects = pd.DataFrame(result["with_aspects"])
    df_w_aspects = pd.DataFrame(result["without_aspects"])
    return df_aspects , df_w_aspects

## 4. Comparing Both Methods

We run both pipelines on the same test set and record their outputs.

A helper function wraps each pipeline call to measure execution time  
and return the results as DataFrames for easy inspection.

In [78]:
df_old_as , df_old_w_as = get_results(analyze_comment_batch , test_comments)

This batch function took 0.1962784279967309 s to be executed


In [82]:
df_old_as.head(10)

,Phrase,Aspects,Label,Score
0,The prices are way too high,[price],negative,0.92
1,Their designs have really improved lately,[design],positive,0.95
2,The store was well staffed during peak hours,[staff],positive,0.88
3,The experience was absolutely priceless,[price],positive,0.98
4,They offer a self-service checkout option,[service],neutral,0.66
5,There is a staffroom on the second floor,[staff],neutral,0.92
6,The service was painfully slow,[service],negative,0.95
7,The design is absolutely beautiful,[design],positive,0.99
8,The quality was surprisingly poor,[quality],negative,0.95
9,The service was not good at all,[service],negative,0.98


### Observation — Keyword Matching

The keyword matching method produces false positives on the substring cases:

- "priceless" → matched as aspect **price**
- "self-service" → matched as aspect **service**
- "staffroom" → matched as aspect **staff**

In each case the aspect keyword appears inside a larger word  
where it carries no relevant meaning.  
The method has no way to distinguish a token boundary from a substring match.

It also misses the inflected forms entirely:  
"deliveries" and "designs" return no aspect,  
because the exact strings "delivery" and "design" do not appear in the phrase.

In [79]:
df_lemma_as , df_lemma_w_as = get_results(analyze_comment_batch , test_comments)

This batch function took 0.1643354219995672 s to be executed


In [83]:
df_lemma_as.head(10)

,Phrase,Aspects,Label,Score
0,The prices are way too high,[price],negative,0.92
1,Their designs have really improved lately,[design],positive,0.95
2,The store was well staffed during peak hours,[staff],positive,0.88
3,The experience was absolutely priceless,[price],positive,0.98
4,They offer a self-service checkout option,[service],neutral,0.66
5,There is a staffroom on the second floor,[staff],neutral,0.92
6,The service was painfully slow,[service],negative,0.95
7,The design is absolutely beautiful,[design],positive,0.99
8,The quality was surprisingly poor,[quality],negative,0.95
9,The service was not good at all,[service],negative,0.98


### Observation — spaCy Lemma Matching

The spaCy lemma method correctly handles inflected forms:

- "prices" → lemma **price** ✓
- "deliveries" → lemma **delivery** ✓
- "designs" → lemma **design** ✓

However, the false positives from the substring cases remain.  
spaCy tokenizes "priceless" as a single token with lemma "priceless",  
so it does not match — this is correct behavior.  
The remaining false positives come from cases like "self-service",  
where spaCy tokenizes "service" as its own token, preserving the match.

Both methods return only the aspect name.  
They tell us what is being discussed, but not how it is described.  
This limitation motivates the next step.

## 5. Opinion Extraction with Dependency Parsing

We introduce a new function that goes beyond detecting aspects  
to also extract the opinion word linked to each aspect.

spaCy's dependency parser assigns each token a grammatical role  
in relation to its head word.  
This allows us to walk the dependency tree and find:

- Adjectives that directly modify an aspect noun (amod relation)
- Adjectives that are predicated about an aspect noun through a verb (acomp, attr relation)

The result is a structured pair: **aspect → opinion word**,  
which captures not just the topic but the judgment expressed about it.

In [25]:
def get_aspect_opinions(phrase):
    doc = nlp(phrase)
    results = []

    for token in doc:
        if token.lemma_.lower() not in ASPECTS:
            continue

        aspect = token.lemma_.lower()
        opinions = []


        for child in token.children:
            if child.dep_ == "amod" and child.pos_ == "ADJ":
                opinions.append(child.text)

        if token.dep_ in ("nsubj", "nsubjpass"):
            for sibling in token.head.children:
                if sibling.dep_ in ("acomp", "attr") and sibling.pos_ == "ADJ":
                    opinions.append(sibling.text)

        results.append({
            "aspect": aspect,
            "opinions": opinions,
            "sentence": token.sent.text.strip()
        })

    return results

In [84]:
results = []
for phrase in test_comments:
    for item in get_aspect_opinions(phrase):
        r = {}
        r["phrase"] = phrase 
        r["aspect"] = item['aspect']
        r["opinion"] = item['opinions']
        results.append(r)
    

In [87]:
df_aspect_opinion = pd.DataFrame(results)
df_aspect_opinion.head(15)

,phrase,aspect,opinion
0,The deliveries were always on time.,delivery,[]
1,The prices are way too high.,price,[high]
2,Their designs have really improved lately.,design,[]
3,The store was well staffed during peak hours.,staff,[]
4,They offer a self-service checkout option.,service,[]
5,The service was painfully slow.,service,[slow]
6,The design is absolutely beautiful.,design,[beautiful]
7,The quality was surprisingly poor.,quality,[poor]
8,The service was not good at all.,service,[good]
9,The performance was never bad in my experience.,performance,[bad]


## 6. Conclusion

This notebook explored two improvements over the basic keyword matching approach.

spaCy lemma matching correctly handles inflected word forms  
and avoids some false positives through proper tokenization.  
For a fixed-list approach, this is a meaningful improvement.

Dependency-based opinion extraction offers stronger aspect recognition  
than keyword matching, and provides an additional layer of structure  
by pairing each aspect with the adjective that describes it.  
This makes it useful as an exploratory tool — giving a quick overview  
of which aspects users mention and what language surrounds them.

However, it has clear limitations that prevent it from scaling to real-world use.  
Opinion identification breaks down on complex sentence structures,  
where the opinion is negated, implicit, or expressed through a verb rather than a direct adjective.  
More critically, the method produces no sentiment classification —  
it cannot distinguish a positive comment from a negative one.  
In practice, when thousands of comments need to be analysed,  
extracting adjectives from dependency trees is not a reliable or sufficient signal.  
A supervised model trained on labelled aspect-sentiment pairs remains the appropriate solution.

In [90]:
#saving results
df_old_as.to_csv("../reports/Data/Aspect_based_analysis_wordMatching.csv")
df_old_w_as.to_csv("../reports/Data/W_Aspect_based_analysis_wordMatching.csv")
df_lemma_as.to_csv("../reports/Data/Aspect_based_analysis_Spacy.csv")
df_lemma_w_as.to_csv("../reports/Data/W_Aspect_based_analysis_Spacy.csv")
df_aspect_opinion.to_csv("../reports/Data/Aspects_Opinion_Spacy.csv")